In [1]:
KLUCZ = "hyppe_Ulen1mjyHUnUrI3fF6vMZzioQlqUZE6P" # CHANGE IT
URL   = "https://hyppe.futura.foundation"

import json, statistics, urllib.error, urllib.request

UA = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/128.0 Safari/537.36")


def wolaj(sciezka, dane=None, limit=900, prob=6):
    """(kod_http, odpowiedz). `dane` != None -> POST z JSON-em.
    """
    import time
    for nr in range(prob):
        z = urllib.request.Request(URL.rstrip('/') + sciezka)
        z.add_header('X-API-Key', KLUCZ)
        z.add_header('User-Agent', UA)
        if dane is not None:
            z.add_header('Content-Type', 'application/json')
            z.data = json.dumps(dane).encode()
        try:
            with urllib.request.urlopen(z, timeout=limit) as o:
                return o.status, json.loads(o.read().decode())
        except urllib.error.HTTPError as e:
            tresc = e.read().decode('utf-8', 'replace')
            ponawialne = (e.code == 503
                          or (e.code == 429 and sciezka != '/wgraj'))
            if not ponawialne or nr == prob - 1:
                return e.code, tresc
            czekaj = float(e.headers.get('Retry-After') or 0) or 0.4 * 2 ** nr
            time.sleep(min(czekaj, 8.0))
        except urllib.error.URLError as e:
            if nr == prob - 1:
                return 0, 'siec: %s' % e
            time.sleep(0.4 * 2 ** nr)


kod_, ja = wolaj('/me')
assert kod_ == 200, ('zly klucz albo brak User-Agent', ja)
print('druzyna    :', ja['druzyna'], '|', ja['uczestnik'])
print('klucz wazny:', ja['wazny_do'], '(%.1f h)' % (ja['wazny_jeszcze_s'] / 3600)
      if ja.get('wazny_jeszcze_s') else 'klucz bez terminu')
print('limity/min :', ja['limity_na_minute'])
print('wgranie za :', ja['zgloszenie_mozliwe_za_s'], 's')


druzyna    : druzyna_05 | druzyna_05_os3
klucz wazny: 2026-08-29 20:15 czasu lokalnego (6.1 h)
limity/min : {'/sedzia': 3000, '/nawigator/mapa': 3000, '/nawigator/edycje': 3000, 'inne': 600}
wgranie za : 0 s


In [2]:
import math
import random
import sys
import time
import pandas as pd

In [3]:
BASES = "ACGT"
LENGTH = 800

_LAST_CALL = [0.0]

In [4]:
def hamming(a, b):
    """Number of positions at which two equal-length sequences differ."""
    return sum(1 for x, y in zip(a, b) if x != y)

def is_valid(seq):
    return len(seq) == LENGTH and set(seq) <= set(BASES)

def _throttle(per_minute):
    """Keep API calls under the 600/min cap. Plain function + global state."""
    if not per_minute or per_minute <= 0:
        return
    gap = 60.0 / per_minute
    dt = time.time() - _LAST_CALL[0]
    if dt < gap:
        time.sleep(gap - dt)
    _LAST_CALL[0] = time.time()

In [9]:
"""
Ewolucja promotorów: DataFrame z rankingiem -> 1000 sekwencji.

Wpisz swój klucz w KLUCZ poniżej i wszystko działa.

    import pandas as pd
    from evolution import evolve

    df = pd.read_csv('Promotory.csv', sep=';')   # posortowany, najlepsze na gorze
    children = evolve(df, k=1000)
    children.to_csv('generation_01.csv', sep=';', index=False)
"""

import json
import math
import random
import time
import urllib.error
import urllib.request

import pandas as pd

KLUCZ = "hyppe_Ulen1mjyHUnUrI3fF6vMZzioQlqUZE6P"
URL = "https://hyppe.futura.foundation"

UA = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/128.0 Safari/537.36")


# ---------------------------------------------------------------- API

def wolaj(sciezka, dane=None):
    """Wywołanie API. Zwraca (kod_http, odpowiedz)."""
    for proba in range(6):
        z = urllib.request.Request(URL + sciezka)
        z.add_header('X-API-Key', KLUCZ)
        z.add_header('User-Agent', UA)
        if dane is not None:
            z.add_header('Content-Type', 'application/json')
            z.data = json.dumps(dane).encode()
        try:
            with urllib.request.urlopen(z, timeout=900) as o:
                return o.status, json.loads(o.read().decode())
        except urllib.error.HTTPError as e:
            if e.code not in (429, 503) or proba == 5:
                return e.code, e.read().decode('utf-8', 'replace')
            time.sleep(0.5 * 2 ** proba)
        except urllib.error.URLError as e:
            if proba == 5:
                return 0, str(e)
            time.sleep(0.5 * 2 ** proba)


# ---------------------------------------------------------------- krok 1: wybór rodziców

def hamming(a, b):
    """Na ilu pozycjach dwie sekwencje się różnią."""
    return sum(1 for x, y in zip(a, b) if x != y)


def select_top(df, fraction=0.2, min_distance=8, id_col='ID', seq_col='sekwencja'):
    """Bierze najlepsze `fraction` rankingu, pomijając sekwencje zbyt podobne
    do już wybranych. Bez tego rodzice szybko stają się klonami jednej linii.

    df musi być posortowany: najlepsza sekwencja na górze.
    Zwraca DataFrame z kolumnami id, sequence.
    """
    ile = int(len(df) * fraction)
    parents = []
    for _, row in df.iterrows():
        if len(parents) >= ile:
            break
        seq = str(row[seq_col]).strip().upper()
        if all(hamming(seq, p['sequence']) >= min_distance for p in parents):
            parents.append({'id': str(row[id_col]), 'sequence': seq})

    print('rodzice: %d z %d (top %.0f%%)' % (len(parents), len(df), fraction * 100))
    return pd.DataFrame(parents)


# ---------------------------------------------------------------- krok 2: ile dzieci

def allocate_children(n_parents, k, tau=None):
    """Rozdziela k dzieci między rodziców wagami exp(-i/tau).
    Najlepszy dostaje najwięcej. Suma zawsze wynosi dokładnie k."""
    if tau is None:
        tau = n_parents / 3

    wagi = [math.exp(-i / tau) for i in range(n_parents)]
    suma = sum(wagi)
    quota = [int(w / suma * k) for w in wagi]

    # rozdaj resztę z zaokrągleń najlepszym rodzicom
    i = 0
    while sum(quota) < k:
        quota[i % n_parents] += 1
        i += 1
    return quota


# ---------------------------------------------------------------- krok 3: Nawigator

def get_map(seq):
    """Mapa pozycji: (wagi gradientu, {pozycja: zalecana zasada}).
    Wagi mówią, gdzie model jest wrażliwy - tam warto mutować."""
    kod, m = wolaj('/nawigator/mapa', {'sekwencja': seq, 'od': 0, 'ile': 800})
    if kod != 200:
        print('  mapa: HTTP %s' % kod)
        return [1.0] * 800, {}

    wagi = [1.0] * 800
    hints = {}
    for p in m['pozycje']:
        i = p['poz'] - 1
        wagi[i] = p['wagaP'] + 0.05
        if p['zmien_na'] != '.':
            hints[i] = p['zmien_na']
    return wagi, hints


def get_edits(seq, ile_potrzeba, options=8):
    """Warianty z /nawigator/edycje. Jedno wywołanie daje `options` sztuk,
    więc wołamy w pętli. Poziom 2 to drobne zmiany, poziom 0 to duże skoki -
    rotujemy je, żeby dzieci miały różny zasięg zmian."""
    variants = []
    widziane = {seq}

    for nr in range(ile_potrzeba // options + 5):
        if len(variants) >= ile_potrzeba:
            break
        poziom = [2, 1, 0][nr % 3]
        kod, e = wolaj('/nawigator/edycje', {
            'sekwencja': seq,
            'poziom': poziom,
            'ile_kodow': random.randint(4, 12),
            'opcji': options,
            'ziarno': random.randint(1, 999999),
        })
        if kod != 200:
            print('  edycje: HTTP %s' % kod)
            break
        for o in e['opcje']:
            s = o['sekwencja']
            if s not in widziane:
                widziane.add(s)
                variants.append(s)
        time.sleep(0.1)   # limit 600/min

    return variants


def mutate(seq, wagi, hints, n_mut):
    """Mutacja punktowa. Pozycje losowane proporcjonalnie do wag z mapy,
    a jeśli Nawigator coś w danym miejscu zaleca, zwykle to bierzemy."""
    s = list(seq)
    for i in random.choices(range(800), weights=wagi, k=n_mut):
        if i in hints and random.random() < 0.6:
            s[i] = hints[i]
        else:
            s[i] = random.choice([b for b in 'ACGT' if b != s[i]])
    return ''.join(s)


# ---------------------------------------------------------------- krok 4: dzieci

def breed(parents, k=1000, keep_elite=5, tau=None):
    """Z rodziców robi k sekwencji.

    Dla każdego rodzica: mapa (raz), warianty z Nawigatora, a na każdy
    wariant jeszcze kilka mutacji punktowych - sam dekoder daje za mało
    różnorodności. keep_elite przepisuje najlepszych rodziców bez zmian,
    żeby pokolenie nie mogło się cofnąć.
    """
    quota = allocate_children(len(parents), k - keep_elite, tau)
    print('podział dzieci:', quota)

    rows = []
    for i in range(keep_elite):
        rows.append({'id': 'elite_%02d' % (i + 1),
                     'sequence': parents.iloc[i]['sequence'],
                     'parent_id': parents.iloc[i]['id'],
                     'changes': 0})

    for rank, (_, p) in enumerate(parents.iterrows()):
        ile = quota[rank]
        seq = p['sequence']
        print('[%d/%d] %s -> %d dzieci' % (rank + 1, len(parents), p['id'], ile))

        wagi, hints = get_map(seq)
        variants = get_edits(seq, ile)

        for j in range(ile):
            baza = variants[j % len(variants)] if variants else seq
            child = mutate(baza, wagi, hints, random.randint(1, 4))
            rows.append({'id': '%s_%03d' % (p['id'], j + 1),
                         'sequence': child,
                         'parent_id': p['id'],
                         'changes': hamming(seq, child)})

    children = pd.DataFrame(rows).drop_duplicates('sequence')
    print('gotowe: %d sekwencji' % len(children))
    return children


def evolve(df, k=1000, fraction=0.2, keep_elite=5):
    """select_top + breed w jednym."""
    parents = select_top(df, fraction)
    return breed(parents, k, keep_elite)


def to_fasta(children):
    return '\n'.join('>%s\n%s' % (r['id'], r['sequence'])
                     for _, r in children.iterrows())

In [10]:
df = pd.read_csv("/home/ejhuus/bromoters/hack-the-BROmoter/HackThePromotor/Promotory.csv", sep=";")

In [13]:
df["ID"] = df.index

In [14]:
df

,nazwa,gatunek,gatunek_krotko,genom,dlugosc,N,sekwencja,ID
0,B10_G000242,B10,B10,B10,800,0,TGCCTGGTTGACACGCTTGACGTTACCAAAACATTTCATATGGTGC...,0
1,F7_G010122,F7,F7,F7,800,0,AATGAAAGATGAGCTAAGAGATTATGGATATCAGACATGCCCTTTC...,1
2,I1237_G008125,I1237,I1237,I1237,800,0,AGGAACTGTCATCAAGATAATCGTGAAGAAAATCATACTTGACGGC...,2
3,MMS1295_G010217,MMS1295,MMS1295,MMS1295,800,0,TGTTGTATTATTTTGTTCTATGGAGCTTATAGATACGTTAATGCAA...,3
4,N1508_G009601,N1508,N1508,N1508,800,0,CTTTGTTCTCCATATTGCGTGGAGCTAGAGTTGCGGTATGTCTCAC...,4
...,...,...,...,...,...,...,...,...
95,F7_G006828,F7,F7,F7,800,0,CACCGGGGCCAGCCCCTCTGTGCCTACTTTGTATACTCAAATTCTC...,95
96,I1237_G010182,I1237,I1237,I1237,800,0,CGAGAAGCTACATGGGTAGCTTAGGGCAGCAGTGGTTATGATGAGC...,96
97,MMS1295_G012219,MMS1295,MMS1295,MMS1295,800,0,TTCGAGAAGCTGGAGTTGAGCTTTTCGAAGAAGAAGCTAGCAGTGA...,97
98,N1508_G000826,N1508,N1508,N1508,800,0,TGATTGATTGGATGTGAGGGCTGTTGATGTGGGTGGTTTTTTTTTG...,98


In [15]:
children = evolve(df, k=1000)

rodzice: 20 z 100 (top 20%)
podział dzieci: [146, 126, 109, 94, 81, 69, 60, 52, 44, 37, 32, 28, 24, 20, 17, 15, 13, 11, 9, 8]
[1/20] 0 -> 146 dzieci
[2/20] 1 -> 126 dzieci


KeyboardInterrupt: 

In [12]:
children

,id,sequence,parent_id,parent_rank,source,changes_vs_parent,length
0,elite_01,TGCCTGGTTGACACGCTTGACGTTACCAAAACATTTCATATGGTGC...,B10_G000242,1,elite,0,800
1,elite_02,AATGAAAGATGAGCTAAGAGATTATGGATATCAGACATGCCCTTTC...,F7_G010122,2,elite,0,800
2,elite_03,AGGAACTGTCATCAAGATAATCGTGAAGAAAATCATACTTGACGGC...,I1237_G008125,3,elite,0,800
3,elite_04,TGTTGTATTATTTTGTTCTATGGAGCTTATAGATACGTTAATGCAA...,MMS1295_G010217,4,elite,0,800
4,elite_05,CTTTGTTCTCCATATTGCGTGGAGCTAGAGTTGCGGTATGTCTCAC...,N1508_G009601,5,elite,0,800
...,...,...,...,...,...,...,...
995,B10_G002171__p20_005,CTGGGCGCATCCTGTTGGTGTGCTGTACCTATACATGTATATAGCT...,B10_G002171,20,edits_L2_k3,2,800
996,B10_G002171__p20_006,CTGGGCGCATCCTGTTGGTGTGCTGTACCTATACATGTATATAGCT...,B10_G002171,20,edits_L1_k9,13,800
997,B10_G002171__p20_007,CTGGGCGCATCCTGTTGGTGTGCTGTACCTGTACATGTAAATAGCT...,B10_G002171,20,edits_L2_k3+mut4,17,800
998,B10_G002171__p20_008,CTGGGCGCATCCTGTTGGTGTGCTGTACCTATACATGTATATAGCT...,B10_G002171,20,edits_L2_k3+mut3,14,800
